### Standard VectorStore Retriever (VectorStoreRetriever)

The VectorStoreRetriever wraps any LangChain vector database (Chroma, FAISS, Qdrant, Pinecone) to expose a standardized Runnable retriever interface.

It is created directly from a vector store instance using the as_retriever method.

### Search Types Demonstrated

* similarity: Retrieves top-k nearest documents.
* similarity_score_threshold: Filters out low-confidence documents using a score threshold.
* mmr: Applies Maximal Marginal Relevance to balance relevance and diversity.

In [ ]:
from dotenv import load_dotenv, find_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

load_dotenv(find_dotenv())

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

docs = [
    Document(page_content="Python is an interpreted, high-level programming language for general-purpose programming.", metadata={"topic": "python"}),
    Document(page_content="Python supports multiple programming paradigms including object-oriented and functional.", metadata={"topic": "python"}),
    Document(page_content="LangChain is a framework designed to simplify creating applications using LLMs.", metadata={"topic": "ai"}),
    Document(page_content="Baking sourdough bread requires flour, water, salt, and wild yeast starter.", metadata={"topic": "baking"})
]

# Ephemeral in-memory Chroma instance
vectorstore = Chroma.from_documents(docs, embeddings)

# 1. Standard Similarity Search Retriever
sim_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})
sim_results = sim_retriever.invoke("Tell me about Python programming")
print("=== 1. Similarity Retriever Results ===")
for i, d in enumerate(sim_results, 1):
    print(f"Doc {i}: {d.page_content}")

# 2. Similarity Score Threshold Retriever
thresh_retriever = vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.3, "k": 3})
thresh_results = thresh_retriever.invoke("Sourdough bread recipe")
print("\n=== 2. Score Threshold Retriever Results ===")
for i, d in enumerate(thresh_results, 1):
    print(f"Doc {i}: {d.page_content}")

# 3. MMR (Maximal Marginal Relevance) Retriever
mmr_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k": 4, "lambda_mult": 0.5})
mmr_results = mmr_retriever.invoke("Python software development")
print("\n=== 3. MMR Retriever Results ===")
for i, d in enumerate(mmr_results, 1):
    print(f"Doc {i}: {d.page_content}")


/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== 1. Similarity Retriever Results ===
Doc 1: Python is an interpreted, high-level programming language for general-purpose programming.
Doc 2: Python supports multiple programming paradigms including object-oriented and functional.

=== 2. Score Threshold Retriever Results ===
Doc 1: Baking sourdough bread requires flour, water, salt, and wild yeast starter.
Doc 2: Python is an interpreted, high-level programming language for general-purpose programming.
Doc 3: Python supports multiple programming paradigms including object-oriented and functional.

=== 3. MMR Retriever Results ===
Doc 1: Python is an interpreted, high-level programming language for general-purpose programming.
Doc 2: Baking sourdough bread requires flour, water, salt, and wild yeast starter.


: 